In [1]:
types = [
    "label", 
    "cot", 
    "checklist_stage1",
    "checklist_stage2",
    "gpt-4o_wCoT",
    "gpt-4o_woCoT",
    "Qwen25-VL-72B-woCoT",
    "Qwen25-VL-72B-wCoT",
    "Qwen25-VL-7B-wCoT",
    "gemini-2.5-flash_woCoT",
    "gemini-2.5-flash_wCoT",
    "qwen3-vl-235b-a22b-thinking_woCoT",
    "gemini-2.5-pro-thinking-1024_woCoT",
    "Qwen25-VL-7B-woCoT",
    "Qwen25-VL-7B-wCoT",
    ]

In [ ]:

def get_binary_metrics_totals(data):
    badcase_tp = 0
    badcase_fp = 0
    badcase_fn = 0
    badcase_support = 0 
    
    goodcase_tp = 0
    goodcase_fp = 0
    goodcase_fn = 0
    goodcase_support = 0 

    for item in data:
        gt_is_badcase = len(item["error_type_list"]) > 0
        pred_is_badcase = len(item["output"]["error_type_list"]) > 0
        
        gt_is_goodcase = not gt_is_badcase
        pred_is_goodcase = not pred_is_badcase
        
        if gt_is_badcase:
            badcase_support += 1
            if pred_is_badcase:
                badcase_tp += 1
            else:
                badcase_fn += 1
        elif pred_is_badcase: 
            badcase_fp += 1

        if gt_is_goodcase:
            goodcase_support += 1
            if pred_is_goodcase:
                goodcase_tp += 1
            else:
                goodcase_fn += 1
        elif pred_is_goodcase: 
            goodcase_fp += 1
            
    return {
        "badcase": {"tp": badcase_tp, "fp": badcase_fp, "fn": badcase_fn, "support": badcase_support},
        "goodcase": {"tp": goodcase_tp, "fp": goodcase_fp, "fn": goodcase_fn, "support": goodcase_support},
        "total_samples": len(data)
    }

def calculate_f1_score(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def calculate_weighted_f1(data) -> float:
    metrics = get_binary_metrics_totals(data)
    
    total_samples = metrics["total_samples"]
    if total_samples == 0:
        return 0.0
    
    badcase_f1 = calculate_f1_score(
        metrics["badcase"]["tp"], 
        metrics["badcase"]["fp"], 
        metrics["badcase"]["fn"]
    )
    weight_badcase = metrics["badcase"]["support"] / total_samples
    
    goodcase_f1 = calculate_f1_score(
        metrics["goodcase"]["tp"], 
        metrics["goodcase"]["fp"], 
        metrics["goodcase"]["fn"]
    )
    weight_goodcase = metrics["goodcase"]["support"] / total_samples
    
    # 3. 计算加权平均 F1
    weighted_f1 = (badcase_f1 * weight_badcase) + (goodcase_f1 * weight_goodcase)
    
    return weighted_f1
    

In [ ]:
def calculate_recall(data):
    recall_list = []
    for item in data:
        pred = [p.strip() for p in item["output"]["error_type_list"]]
        gt = [p.strip() for p in item["error_type_list"]]
        
        if len(gt) == 0:
            if len(pred) > 0:
                recall = 0.0
            else:
                recall = 1.0 
        else:
            tp = 0
            for p in pred:
                if p in gt:
                    tp += 1
            recall = tp / len(gt)
        
        recall_list.append(recall)
    return sum(recall_list) / len(recall_list)

def calculate_precision(data):
    precision_list = []
    for item in data:
        pred = [p.strip() for p in item["output"]["error_type_list"]]
        gt = [p.strip() for p in item["error_type_list"]]
        
        if len(pred) == 0:
            if len(gt) == 0:
                precision = 1.0
            else:
                precision = 0.0 
        else:
            tp = 0
            for p in pred:
                if p in gt:
                    tp += 1
            precision = tp / len(pred)
        
        precision_list.append(precision)
    return sum(precision_list) / len(precision_list) if len(precision_list) > 0 else 0.0

def calculate_f1(data):
    f1_list = []
    for item in data:
        pred = [p.strip() for p in item["output"]["error_type_list"]]
        gt = [p.strip() for p in item["error_type_list"]]
        
        if len(pred) == 0:
            if len(gt) == 0:
                precision = 1.0
                recall = 1.0
            else:
                precision = 0.0  
                recall = 0.0
        else:
            tp = 0
            for p in pred:
                if p in gt:
                    tp += 1
            precision = tp / len(pred)
            recall = tp / len(gt) if len(gt) > 0 else 0.0

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * precision * recall / (precision + recall)
        
        f1_list.append(f1)
    return sum(f1_list) / len(f1_list) if len(f1_list) > 0 else 0.0

In [ ]:
import json
import pandas as pd
from tabulate import tabulate

results_dict = {}
for type in types:
    json_path = f"../results/{type}_test_results.json"
    with open(json_path, "r") as f:
        data = json.load(f)
    
    text_data = [item for item in data if item["category"] != "table" and "equation" not in item["category"]]
    table_data = [item for item in data if item["category"] == "table"]
    formula_data = [item for item in data if "equation" in item["category"]]
    
    text_goodcase_data = [item for item in text_data if len(item["error_type_list"]) == 0]
    text_badcase_data = [item for item in text_data if len(item["error_type_list"]) != 0]
    
    table_goodcase_data = [item for item in table_data if len(item["error_type_list"]) == 0]
    table_badcase_data = [item for item in table_data if len(item["error_type_list"]) != 0]
    
    formula_goodcase_data = [item for item in formula_data if len(item["error_type_list"]) == 0]
    formula_badcase_data = [item for item in formula_data if len(item["error_type_list"]) != 0]
    
    
    results_dict[type] = {
        "text": {
            "case_f1": calculate_weighted_f1(text_data),
            "precision": calculate_precision(text_data),
            "f1": calculate_f1(text_data),
            "recall": calculate_recall(text_data)
        },
        "table": {
            "case_f1": calculate_weighted_f1(table_data),
            "precision": calculate_precision(table_data),
            "f1": calculate_f1(table_data),
            "recall": calculate_recall(table_data)
        },
        "formula": {
            "case_f1": calculate_weighted_f1(formula_data),
            "precision": calculate_precision(formula_data),
            "f1": calculate_f1(formula_data),
            "recall": calculate_recall(formula_data)
        },
        "all": {
            "case_f1": calculate_weighted_f1(data),
            "precision": calculate_precision(data),
            "f1": calculate_f1(data),
            "recall": calculate_recall(data)
        }
    }


df_data = []
for type_name, metrics in results_dict.items():
    for category, scores in metrics.items():
        row = {
            "Type": type_name,
            "Category": category.capitalize(), 
            "Recall": scores["recall"],
            "F1": scores["f1"],
            "precision": scores["precision"],
            "case_f1": scores["case_f1"],
        }
        df_data.append(row)


df = pd.DataFrame(df_data)


df["precision"] = df["precision"].map('{:.4f}'.format)
df["F1"] = df["F1"].map('{:.4f}'.format)
df["Recall"] = df["Recall"].map('{:.4f}'.format)
df["case_f1"] = df["case_f1"].map('{:.4f}'.format)




In [ ]:

df_data = []
for type_name, metrics in results_dict.items():
    for category, scores in metrics.items():
        row = {
            "Type": type_name,
            "Category": category.capitalize(),
            "Recall": scores["recall"],
            "F1": scores["f1"],
            "precision": scores["precision"],
            "case_f1": scores["case_f1"],
        }
        df_data.append(row)

df = pd.DataFrame(df_data)

df["precision"] = (df["precision"] * 100).map('{:.2f}'.format)
df["F1"] = (df["F1"] * 100).map('{:.2f}'.format)
df["Recall"] = (df["Recall"] * 100).map('{:.2f}'.format)
df["case_f1"] = (df["case_f1"] * 100).map('{:.2f}'.format)


df_melted = df.melt(
    id_vars=["Type", "Category"],
    value_vars=["Recall", "F1", "precision", "case_f1"],  
    var_name="Metric",  
    value_name="Score"  
)

df_pivoted = df_melted.pivot(
    index="Type",        
    columns=["Category", "Metric"], 
    values="Score"       
)


categories = ['Text', 'Table', 'Formula', 'All'] 
metrics_order = ['Recall', 'F1', 'precision', 'case_f1']


model_order = [
    "Qwen25-VL-7B-woCoT",
    "Qwen25-VL-7B-wCoT",
    "Qwen25-VL-72B-woCoT",
    "Qwen25-VL-72B-wCoT",
    "gpt-4o_wCoT",
    "gpt-4o_woCoT",
    "gemini-2.5-flash_woCoT",
    "gemini-2.5-flash_wCoT",
    "qwen3-vl-235b-a22b-thinking_woCoT",
    "gemini-2.5-pro-thinking-1024_woCoT",
    "label", 
    "cot", 
    "checklist_stage1",
    "checklist_stage2",    
    ]

df_pivoted = df_pivoted.reindex(model_order)

# 创建一个新的列顺序列表
new_column_order = []
for cat in categories:
    for metric in metrics_order:
        if (cat, metric) in df_pivoted.columns:
            new_column_order.append((cat, metric))

df_final = df_pivoted[new_column_order]


df_final.columns.names = ['Category', 'Metric']

df_final.style.set_properties(**{'text-align': 'center'})